In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# Configuración visual para un reporte profesional
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams['figure.figsize'] = (14, 7)

# ==========================================
# 1. CARGA Y LIMPIEZA DE DATOS (10 AÑOS)
# ==========================================
print("Cargando y consolidando archivos de Excel (2015-2024)...")
archivos = [f'BD_{year}.xlsx' for year in range(2015, 2025)]

df_list = []
col_estacion = 'STATION'
col_fecha = 'DATE'

for archivo in archivos:
    if os.path.exists(archivo):
        df_temp = pd.read_excel(archivo)
        if col_estacion in df_temp.columns:
            df_temp = df_temp[df_temp[col_estacion].astype(str).str.contains('PIN', case=False, na=False)]
            df_list.append(df_temp)
    else:
        print(f"  [X] Archivo no encontrado: {archivo}")

if not df_list:
    raise ValueError("No se encontraron datos. Sube los archivos al entorno.")

df = pd.concat(df_list, ignore_index=True)
df[col_fecha] = pd.to_datetime(df[col_fecha])
df.set_index(col_fecha, inplace=True)
df.sort_index(inplace=True)

# Extraer variables temporales
df['AÑO'] = df.index.year
df['MES'] = df.index.month
df['HORA'] = df.index.hour

# Listas de variables dinámicas (solo usa las que existan en tu BD)
contaminantes = [c for c in ['O3', 'NO', 'NO2', 'NOX', 'SO2', 'CO', 'PM10', 'PM2.5'] if c in df.columns]
meteorologia = [c for c in ['IT', 'ET', 'RH', 'WS', 'WD', 'PP', 'ATM', 'RS', 'UVI'] if c in df.columns]

# ---> CORRECCIÓN AQUÍ: Forzar conversión a números y convertir 'IO' u otros textos a NaN <---
for col in contaminantes + meteorologia:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# ==========================================
# 2. DISPONIBILIDAD Y LIMITACIONES DE DATOS
# ==========================================
print("\n--- EVALUANDO CALIDAD Y DISPONIBILIDAD DE DATOS ---")
disponibilidad = df.groupby('AÑO').count() / df.groupby('AÑO').size().values.reshape(-1, 1) * 100
disponibilidad = disponibilidad[contaminantes + meteorologia].round(2)

limitaciones = disponibilidad[disponibilidad < 75].dropna(how='all')

print("LIMITACIÓN DEL ANÁLISIS (Reportar en documento final):")
if not limitaciones.empty:
    print("Las siguientes variables presentan menos del 75% de suficiencia anual, lo que limita la robustez de sus tendencias históricas:")
    for año, fila in limitaciones.iterrows():
        vars_afectadas = fila.dropna().index.tolist()
        print(f" - {año}: {', '.join(vars_afectadas)}")
else:
    print("Todos los años y variables cumplen con más del 75% de disponibilidad.")

# Gráfica 1: Mapa de calor de disponibilidad
plt.figure(figsize=(14, 8))
sns.heatmap(disponibilidad.T, annot=True, cmap="RdYlGn", fmt=".0f", vmin=0, vmax=100)
plt.title("Disponibilidad de Datos por Año y Variable (%) - Las Pintas")
plt.tight_layout()
plt.savefig("1_Disponibilidad_Datos.png")
plt.close()

# ==========================================
# 3. TENDENCIAS DE LARGO PLAZO Y COMPORTAMIENTO ANUAL
# ==========================================
promedios_anuales = df.groupby('AÑO')[contaminantes].mean(numeric_only=True)

# Gráfica 2: Tendencias históricas
plt.figure(figsize=(12, 6))
for contam in ['PM10', 'PM2.5', 'O3']:
    if contam in promedios_anuales.columns:
        sns.lineplot(data=promedios_anuales, x=promedios_anuales.index, y=contam, marker='o', label=contam)
plt.title("Tendencias de Largo Plazo (Promedios Anuales) - Las Pintas")
plt.ylabel("Concentración Promedio")
plt.xlabel("Año")
plt.xticks(promedios_anuales.index)
plt.legend()
plt.tight_layout()
plt.savefig("2_Tendencias_Largo_Plazo.png")
plt.close()

# ==========================================
# 4. EVENTOS ATÍPICOS (OUTLIERS)
# ==========================================
if 'PM10' in df.columns:
    plt.figure(figsize=(14, 6))
    sns.boxplot(x='AÑO', y='PM10', data=df, fliersize=3, palette="Set3")
    plt.title("Distribución Anual y Eventos Atípicos (PM10) - Las Pintas")
    plt.ylabel("Concentración (µg/m³)")
    plt.xlabel("Año")
    plt.tight_layout()
    plt.savefig("3_Eventos_Atipicos.png")
    plt.close()

# ==========================================
# 5. MESES Y PERIODOS CRÍTICOS (ESTACIONALIDAD)
# ==========================================
promedios_mensuales = df.groupby('MES')[contaminantes].mean(numeric_only=True)

if 'PM10' in df.columns:
    meses_criticos = df.pivot_table(values='PM10', index='MES', columns='AÑO', aggfunc='mean')
    plt.figure(figsize=(12, 6))
    sns.heatmap(meses_criticos, cmap="YlOrRd", annot=True, fmt=".0f")
    plt.title("Periodos Críticos: Concentración Promedio Mensual de PM10")
    plt.ylabel("Mes (1=Ene, 12=Dic)")
    plt.xlabel("Año")
    plt.tight_layout()
    plt.savefig("4_Meses_Criticos.png")
    plt.close()

# ==========================================
# 6. RELACIONES CON VARIABLES METEOROLÓGICAS
# ==========================================
cols_corr = contaminantes + meteorologia
matriz_corr = df[cols_corr].corr(numeric_only=True)

plt.figure(figsize=(14, 10))
sns.heatmap(matriz_corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1, fmt=".2f", center=0)
plt.title("Correlación: Contaminantes vs Variables Meteorológicas (10 Años)")
plt.tight_layout()
plt.savefig("5_Correlaciones_Meteorologicas.png")
plt.close()

# ==========================================
# 7. CAMBIOS EN CATEGORÍAS DE CALIDAD DEL AIRE
# ==========================================
daily = pd.DataFrame(index=df.resample('D').size().index)
daily['AÑO'] = daily.index.year

if 'PM10' in df.columns:
    daily['PM10_24H'] = df['PM10'].rolling(window=24, min_periods=18).mean().resample('D').max()
    daily['Cat_PM10'] = pd.cut(daily['PM10_24H'],
                               bins=[0, 50, 75, 155, 235, 9999],
                               labels=['Buena', 'Aceptable', 'Mala', 'Muy Mala', 'Extremadamente Mala'])

if 'PM10' in df.columns:
    cat_anual = daily.groupby(['AÑO', 'Cat_PM10']).size().unstack(fill_value=0)
    cat_anual_pct = cat_anual.div(cat_anual.sum(axis=1), axis=0) * 100

    colores = {'Buena':'#00E400', 'Aceptable':'#FFFF00', 'Mala':'#FF7E00', 'Muy Mala':'#FF0000', 'Extremadamente Mala':'#8F3F97'}
    cat_anual_pct.plot(kind='bar', stacked=True, color=[colores.get(c, '#333') for c in cat_anual_pct.columns], figsize=(14, 6))
    plt.title("Evolución de Categorías de Calidad del Aire (Días por Año) - PM10")
    plt.ylabel("Porcentaje de Días (%)")
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig("6_Categorias_Calidad_Aire.png")
    plt.close()

# ==========================================
# 8. EXPORTACIÓN DEL REPORTE ESTADÍSTICO
# ==========================================
archivo_salida = "Analisis_Integral_10_Anos_LasPintas.xlsx"
print(f"\n--> Exportando métricas y estadísticas a {archivo_salida}...")

with pd.ExcelWriter(archivo_salida, engine='openpyxl') as writer:
    disponibilidad.to_excel(writer, sheet_name='Disponibilidad_Pct')
    promedios_anuales.to_excel(writer, sheet_name='Tendencias_Anuales')
    promedios_mensuales.to_excel(writer, sheet_name='Estacionalidad_Mensual')
    matriz_corr.to_excel(writer, sheet_name='Correlaciones_Pearson')

    if 'PM10' in df.columns:
        cat_anual_pct.to_excel(writer, sheet_name='Evolucion_Categorias_Pct')

print("¡Análisis completado con éxito! Se generaron 6 gráficas (.png) y 1 archivo de Excel.")

Cargando y consolidando archivos de Excel (2015-2024)...

--- EVALUANDO CALIDAD Y DISPONIBILIDAD DE DATOS ---
LIMITACIÓN DEL ANÁLISIS (Reportar en documento final):
Las siguientes variables presentan menos del 75% de suficiencia anual, lo que limita la robustez de sus tendencias históricas:
 - 2015: NO, NOX, SO2, PM2.5, IT, ATM, UVI
 - 2016: NO, NOX, SO2, PM10, PM2.5, IT, PP, ATM, UVI
 - 2017: NO, NOX, SO2, PM2.5, IT, ATM, UVI
 - 2018: NO, NOX, SO2, PM10, PM2.5, RS, UVI
 - 2019: NO, NO2, NOX, PM2.5, PP, UVI
 - 2020: NO, NO2, NOX, SO2, PM10, PM2.5, RS, UVI
 - 2021: NO, NO2, NOX, PM10, PM2.5, UVI
 - 2022: NO, NO2, NOX, SO2, CO, PM2.5, ET, WD, UVI
 - 2023: NO, NO2, NOX, SO2, CO, PM2.5, ATM, RS, UVI
 - 2024: NO, NO2, NOX, SO2, CO, RH, WS, WD, ATM, RS, UVI

--> Exportando métricas y estadísticas a Analisis_Integral_10_Anos_LasPintas.xlsx...
¡Análisis completado con éxito! Se generaron 6 gráficas (.png) y 1 archivo de Excel.
